# PPML (Poisson Pseudo-Maximum Likelihood): polars_reg vs R Verification

This notebook verifies that `polars_reg.ppml()` produces identical results to R's
`glm(family=poisson())` and `fixest::fepois()` using the **RecreationDemand** dataset
from the `AER` package (659 observations).

Tests cover:
1. PPML with default sandwich (HC1) SEs vs `sandwich::vcovHC(type="HC1")`
2. PPML with iid (Poisson assumption) SEs vs `vcov(model)`
3. PPML with more covariates
4. PPML vs `fixest::fepois()` with robust SEs
5. Full summary display

In [ ]:
import sys
import tempfile
from pathlib import Path

# Ensure polars_reg and the helper are importable
REPO = Path.home() / "research" / "polars_reg"
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks" / "verification"))

import polars as pl
import polars_reg as pr
import r_helper

## Load and prepare data

The `RecreationDemand` dataset from the `AER` package contains 659 observations on
recreational trip demand. The count outcome `trips` is non-negative, making it
suitable for PPML estimation.

The `ski` column is a factor ("yes"/"no") in R. We convert it to a binary integer
`ski_bin` for exact comparison between polars_reg and R.

In [ ]:
# Load RecreationDemand from R's AER package
df = r_helper.load_r_dataset(
    "RecreationDemand",
    package="AER",
    extra_code='df$ski_bin <- as.integer(df$ski == "yes"); df$userfee_bin <- as.integer(df$userfee == "yes")'
)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
print(f"trips range: [{df['trips'].min()}, {df['trips'].max()}]")
print(f"trips mean: {df['trips'].mean():.2f}")
df.head(5)

In [ ]:
# Save CSV for R scripts
csv_path = tempfile.mktemp(suffix=".csv")
df.to_pandas().to_csv(csv_path, index=False)
print(f"CSV saved to: {csv_path}")

## 1. PPML with default sandwich (HC1) standard errors

Compare `polars_reg.ppml()` (default `vcov="HC1"`) against R's
`glm(family=poisson())` with `sandwich::vcovHC(type="HC1")`.

In [ ]:
result_hc1 = pr.ppml("trips ~ quality + income + userfee_bin + costC", data=df)
print(f"N={result_hc1.n_obs}, vcov_type={result_hc1.vcov_type}")
print(f"Pseudo R2={result_hc1.r_squared:.6f}")

In [ ]:
r_script = f'''
library(sandwich)
df <- read.csv("{csv_path}")
model <- glm(trips ~ quality + income + userfee_bin + costC,
             data=df, family=poisson())
vcov_mat <- vcovHC(model, type="HC1")
{r_helper.R_EXTRACT}
'''
r_hc1 = r_helper.run_r_regression(r_script)
comp_hc1 = r_helper.compare(result_hc1, r_hc1, rtol=1e-6, se_rtol=1e-5, label="PPML HC1")
comp_hc1

## 2. PPML with iid (Poisson assumption) standard errors

Compare `polars_reg.ppml(vcov="iid")` against R's default `vcov(model)` from
`glm(family=poisson())`, which assumes the Poisson variance structure.

In [ ]:
result_iid = pr.ppml("trips ~ quality + income + userfee_bin + costC", data=df, vcov="iid")
print(f"N={result_iid.n_obs}, vcov_type={result_iid.vcov_type}")

In [ ]:
r_script = f'''
df <- read.csv("{csv_path}")
model <- glm(trips ~ quality + income + userfee_bin + costC,
             data=df, family=poisson())
vcov_mat <- vcov(model)
{r_helper.R_EXTRACT}
'''
r_iid = r_helper.run_r_regression(r_script)
comp_iid = r_helper.compare(result_iid, r_iid, rtol=1e-6, label="PPML iid")
comp_iid

## 3. PPML with more covariates

Include all available numeric covariates plus the binary ski indicator.
Compare HC1 sandwich SEs.

In [ ]:
result_full = pr.ppml(
    "trips ~ quality + ski_bin + income + userfee_bin + costC + costS + costH",
    data=df,
)
print(f"N={result_full.n_obs}, k={result_full.k}")
print(f"Pseudo R2={result_full.r_squared:.6f}")

In [ ]:
r_script = f'''
library(sandwich)
df <- read.csv("{csv_path}")
model <- glm(trips ~ quality + ski_bin + income + userfee_bin + costC + costS + costH,
             data=df, family=poisson())
vcov_mat <- vcovHC(model, type="HC1")
{r_helper.R_EXTRACT}
'''
r_full = r_helper.run_r_regression(r_script)
# Slightly wider SE tolerance for the full model (numerical precision at 7 variables)
comp_full = r_helper.compare(result_full, r_full, rtol=1e-6, se_rtol=5e-5, label="PPML full model HC1")
comp_full

## 4. PPML vs fixest::fepois with robust SEs

Compare `polars_reg.ppml(vcov="HC1")` against `fixest::fepois(vcov="hetero")`.
fixest is widely used for PPML in applied econometrics.

In [ ]:
result_fixest = pr.ppml("trips ~ quality + income + userfee_bin + costC", data=df, vcov="HC1")
print(f"N={result_fixest.n_obs}, vcov_type={result_fixest.vcov_type}")

In [ ]:
r_script = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- fepois(trips ~ quality + income + userfee_bin + costC,
                data=df, vcov="hetero")
vcov_mat <- vcov(model)
{r_helper.R_EXTRACT}
'''
r_fixest = r_helper.run_r_regression(r_script)
comp_fixest = r_helper.compare(result_fixest, r_fixest, rtol=1e-6, se_rtol=1e-4, label="PPML vs fixest")
comp_fixest

## 5. Summary display

Full regression output for inspection.

In [ ]:
print(result_hc1.summary())

In [ ]:
result_hc1.coef_table()

In [ ]:
print(result_full.summary())

In [ ]:
result_full.coef_table()

<cell_type>markdown</cell_type>## Summary

All PPML tests compare `polars_reg.ppml()` against R's `glm(family=poisson())` and `fixest::fepois()`:

| Test | SE type | R reference | Coef tol | SE tol | Status |
|------|---------|-------------|----------|--------|--------|
| Basic PPML | HC1 (sandwich) | `sandwich::vcovHC(type="HC1")` | 1e-6 | 1e-5 | see above |
| PPML iid | Poisson VCV | `vcov(glm(...))` | 1e-6 | 1e-6 | see above |
| Full model | HC1 (sandwich) | `sandwich::vcovHC(type="HC1")` | 1e-6 | 5e-5 | see above |
| vs fixest | hetero | `fixest::fepois(vcov="hetero")` | 1e-6 | 1e-4 | see above |